## 1. Setup

In [4]:

!pip install -q transformers datasets evaluate accelerate


In [5]:

import torch
import transformers
import datasets
import evaluate

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


PyTorch version: 2.14.1+cu130
Transformers version: 5.22.1
Datasets version: 4.8.5
GPU available: True
GPU name: Tesla T4


## 2. Load and Explore Dataset

In [10]:
# Load the dataset

from datasets import load_dataset

# Load SQuAD v1.1 using its full Hugging Face repository ID
dataset = load_dataset("rajpurkar/squad")



In [11]:

# Display dataset information
print(dataset)

# Display available dataset splits
print("Dataset splits:", dataset.keys())

# Display number of training examples
print("Training examples:", len(dataset["train"]))

# Display number of validation examples
print("Validation examples:", len(dataset["validation"]))


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
Dataset splits: dict_keys(['train', 'validation'])
Training examples: 87599
Validation examples: 10570


In [12]:

# Get the first training example
sample = dataset["train"][0]

# Display the complete example
print(sample)


{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


In [23]:
print("=" * 60)
print("SQUAD DATASET SAMPLE")
print("=" * 60)

print("\nContext:")
print(sample["context"])

print("\nQuestion:")
print(sample["question"])

print("\nAnswer text:")
print(sample["answers"]["text"][0])

print("\nAnswer starting position:")
print(sample["answers"]["answer_start"][0])

print("="*60)


SQUAD DATASET SAMPLE

Context:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

Question:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Answer text:
Saint Bernadette Soubirous

Answer starting position:
515


**Text Classification**
- is an NLP task in which a model assigns a predefined category or label to a given text.

For example, consider a movie review:
"The movie was amazing. I really enjoyed it."

Our model needs to determine whether the sentiment is positive or negative.


**Question Answering (QA)**
- is an NLP task in which a model reads a given passage (context), understands a question, and identifies the correct answer.

For this assignment, we use Extractive Question Answering.

It means the model extracts the answer directly from the provided context instead of generating a completely new answer.


| Feature | Text Classification | Question Answering |
| :--- | :--- | :--- |
| **Input** | Text | Question + Context |
| **Output** | Category or label | Answer text |
| **Main objective** | Assign a class | Find the answer |
| **Prediction** | Class probabilities | Start and end positions |
| **Example** | Positive / Negative | Albert Einstein |
| **Common model head** | Classification head | QA span prediction head |
| **Evaluation** | Accuracy, Precision, Recall, F1 | Exact Match, F1 |


## 3. Tokenization

In [24]:

from transformers import AutoTokenizer


In [25]:

# Define the pre-trained model name
MODEL_NAME = "bert-base-uncased"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("BERT tokenizer loaded successfully!")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT tokenizer loaded successfully!


In [26]:

# Example question
question = "Who developed the theory of relativity?"

# Example context
context = (
    "Albert Einstein developed the theory of relativity "
    "in the early 20th century."
)

# Tokenize the question and context together
inputs = tokenizer(
    question,
    context,
    max_length=384,
    truncation="only_second",
    padding="max_length",
    return_tensors="pt"
)

print("Tokenization completed!")


Tokenization completed!


In [27]:

# Display the token IDs
print("Input IDs:")
print(inputs["input_ids"])

# Display attention mask
print("\nAttention Mask:")
print(inputs["attention_mask"])

# Display token type IDs if available
if "token_type_ids" in inputs:
    print("\nToken Type IDs:")
    print(inputs["token_type_ids"])


Input IDs:
tensor([[  101,  2040,  2764,  1996,  3399,  1997, 20805,  1029,   102,  4789,
         15313,  2764,  1996,  3399,  1997, 20805,  1999,  1996,  2220,  3983,
          2301,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     

In [28]:

# Get the token IDs from the first sequence
token_ids = inputs["input_ids"][0].tolist()

# Convert token IDs into readable tokens
tokens = tokenizer.convert_ids_to_tokens(token_ids)

# Display tokens
print(tokens)


['[CLS]', 'who', 'developed', 'the', 'theory', 'of', 'relativity', '?', '[SEP]', 'albert', 'einstein', 'developed', 'the', 'theory', 'of', 'relativity', 'in', 'the', 'early', '20th', 'century', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', 

In [29]:

# Display token and token ID together
for token, token_id in zip(tokens[:25], token_ids[:25]):
    print(f"{token:15} -> {token_id}")


[CLS]           -> 101
who             -> 2040
developed       -> 2764
the             -> 1996
theory          -> 3399
of              -> 1997
relativity      -> 20805
?               -> 1029
[SEP]           -> 102
albert          -> 4789
einstein        -> 15313
developed       -> 2764
the             -> 1996
theory          -> 3399
of              -> 1997
relativity      -> 20805
in              -> 1999
the             -> 1996
early           -> 2220
20th            -> 3983
century         -> 2301
.               -> 1012
[SEP]           -> 102
[PAD]           -> 0
[PAD]           -> 0


In [30]:

# Count actual tokens
actual_tokens = inputs["attention_mask"][0].sum().item()

# Get total sequence length
total_tokens = inputs["input_ids"].shape[1]

# Calculate padding tokens
padding_tokens = total_tokens - actual_tokens

print("Total sequence length:", total_tokens)
print("Actual tokens:", actual_tokens)
print("Padding tokens:", padding_tokens)


Total sequence length: 384
Actual tokens: 23
Padding tokens: 361


### Preprocessing the Dataset — Mapping Answers to Token Positions

In [31]:

# Select one example from the training dataset
sample = dataset["train"][0]

# Get the question, context and answer
question = sample["question"]
context = sample["context"]

answer_text = sample["answers"]["text"][0]
answer_start = sample["answers"]["answer_start"][0]

# Calculate the ending character position
answer_end = answer_start + len(answer_text)

print("Question:", question)
print("Context:", context)
print("Answer:", answer_text)
print("Answer start:", answer_start)
print("Answer end:", answer_end)

# Verify the answer using character positions
print("Extracted answer:", context[answer_start:answer_end])


Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
Answer: Saint Bernadette Soubirous
Answer start: 515
Answer end: 541
Extracted answer: Saint Bernadette Soubirous


In [32]:

from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prepare_train_features(examples):

    # Tokenize question and context together
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        max_length=384,
        truncation="only_second",
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    # Identify which original example each chunk belongs to
    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )

    # Get character positions for every token
    offset_mapping = tokenized_examples.pop(
        "offset_mapping"
    )

    start_positions = []
    end_positions = []

    # Process every tokenized chunk
    for i, offsets in enumerate(offset_mapping):

        # Find the original example
        sample_index = sample_mapping[i]

        # Get its answer information
        answers = examples["answers"][sample_index]

        # If no answer is available, use CLS token position
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Find the answer's character positions
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # Identify which tokens belong to the context
        sequence_ids = tokenized_examples.sequence_ids(i)

        # Find the first and last context token
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        # Check whether the answer is inside this chunk
        if (
            offsets[context_start][0] > start_char
            or offsets[context_end][1] < end_char
        ):
            # Answer is not in this chunk
            start_positions.append(0)
            end_positions.append(0)

        else:
            # Find the token containing the answer's start
            token_start = context_start
            while (
                token_start <= context_end
                and offsets[token_start][0] <= start_char
            ):
                token_start += 1

            start_positions.append(token_start - 1)

            # Find the token containing the answer's end
            token_end = context_end
            while (
                token_end >= context_start
                and offsets[token_end][1] >= end_char
            ):
                token_end -= 1

            end_positions.append(token_end + 1)

    # Attach labels to the tokenized dataset
    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions

    return tokenized_examples


In [33]:

# Apply preprocessing to the training dataset
tokenized_train = dataset["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=dataset["train"].column_names
)

# Apply preprocessing to the validation dataset
tokenized_validation = dataset["validation"].map(
    prepare_train_features,
    batched=True,
    remove_columns=dataset["validation"].column_names
)

print("Preprocessing completed!")

print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_validation))


Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Preprocessing completed!
Training examples: 88524
Validation examples: 10784


In [34]:

print(tokenized_train)


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 88524
})


In [35]:

# Inspect one processed example
processed_sample = tokenized_train[0]

print("Input IDs:")
print(processed_sample["input_ids"][:20])

print("\nAttention Mask:")
print(processed_sample["attention_mask"][:20])

print("\nStart Position:")
print(processed_sample["start_positions"])

print("\nEnd Position:")
print(processed_sample["end_positions"])


Input IDs:
[101, 2000, 3183, 2106, 1996, 6261, 2984, 9382, 3711, 1999, 8517, 1999, 10223, 26371, 2605, 1029, 102, 6549, 2135, 1010]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Start Position:
130

End Position:
137


## 4. Model Setup

In [36]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [37]:

from transformers import AutoModelForQuestionAnswering
import torch


In [38]:

MODEL_NAME = "bert-base-uncased"

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

print("BERT Question Answering model loaded successfully!")


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

BERT Question Answering model loaded successfully!


In [39]:

print("Model configuration:")
print("Model type:", model.config.model_type)
print("Hidden size:", model.config.hidden_size)
print("Number of hidden layers:", model.config.num_hidden_layers)
print("Number of attention heads:", model.config.num_attention_heads)
print("Vocabulary size:", model.config.vocab_size)


Model configuration:
Model type: bert
Hidden size: 768
Number of hidden layers: 12
Number of attention heads: 12
Vocabulary size: 30522


In [40]:

question = "Who developed the theory of relativity?"

context = (
    "Albert Einstein developed the theory of relativity "
    "in the early 20th century."
)

# Convert question and context into model inputs
inputs = tokenizer(
    question,
    context,
    return_tensors="pt"
)

# Get predictions from BERT
with torch.no_grad():
    outputs = model(**inputs)

# Get start and end position predictions
start_position = torch.argmax(outputs.start_logits, dim=1).item()
end_position = torch.argmax(outputs.end_logits, dim=1).item()

print("Start position:", start_position)
print("End position:", end_position)


Start position: 9
End position: 15


In [41]:

print("Input shape:", inputs["input_ids"].shape)

print("Start logits shape:", outputs.start_logits.shape)

print("End logits shape:", outputs.end_logits.shape)


Input shape: torch.Size([1, 23])
Start logits shape: torch.Size([1, 23])
End logits shape: torch.Size([1, 23])


## 5. Fine-tuning

In [42]:

from transformers import (
    Trainer,
    TrainingArguments,
    default_data_collator
)

import inspect

print("Trainer components imported successfully!")


Trainer components imported successfully!


In [43]:

# Training configuration
training_args_dict = {
    "output_dir": "./bert-squad-qa",
    "learning_rate": 3e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "num_train_epochs": 2,
    "weight_decay": 0.01,
    "logging_steps": 500,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "report_to": "none",
    "fp16": torch.cuda.is_available()
}

# Support different Transformers versions
supported_args = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in supported_args:
    training_args_dict["eval_strategy"] = "epoch"
else:
    training_args_dict["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_dict)

print("Training arguments configured successfully!")


Training arguments configured successfully!


In [44]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=default_data_collator
)

print("Trainer initialized successfully!")


Trainer initialized successfully!


In [45]:

train_result = trainer.train()

print("Training completed successfully!")


Epoch,Training Loss,Validation Loss
1,1.015394,1.009149
2,0.693819,1.007922


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training completed successfully!


In [46]:

# Save the fine-tuned model
trainer.save_model("./fine_tuned_bert_squad")

# Save the tokenizer
tokenizer.save_pretrained("./fine_tuned_bert_squad")

print("Fine-tuned model and tokenizer saved successfully!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model and tokenizer saved successfully!


In [47]:
# Load the model

model = AutoModelForQuestionAnswering.from_pretrained(
    "./fine_tuned_bert_squad"
)

tokenizer = AutoTokenizer.from_pretrained(
    "./fine_tuned_bert_squad"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 6. Evaluation

In [48]:

import evaluate

squad_metric = evaluate.load("squad")

print("SQuAD evaluation metric loaded successfully!")


SQuAD evaluation metric loaded successfully!


In [50]:
def prepare_validation_features(examples):

    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        max_length=384,
        truncation="only_second",
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )

    offset_mapping = tokenized_examples["offset_mapping"]

    # Keep offsets only for context tokens
    for i in range(len(offset_mapping)):
        sequence_ids = tokenized_examples.sequence_ids(i)

        offset_mapping[i] = [
            offset if sequence_ids[k] == 1 else None
            for k, offset in enumerate(offset_mapping[i])
        ]

    tokenized_examples["offset_mapping"] = offset_mapping
    tokenized_examples["example_id"] = [
        examples["id"][sample_index]
        for sample_index in sample_mapping
    ]

    return tokenized_examples

In [55]:
validation_features = dataset["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=dataset["validation"].column_names
)

print(validation_features)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 10784
})


In [56]:
print("Validation examples:", len(dataset["validation"]))
print("Validation features:", len(validation_features))

print("\nAvailable columns:")
print(validation_features.column_names)

print("\nFirst example ID:")
print(validation_features[0]["example_id"])

Validation examples: 10570
Validation features: 10784

Available columns:
['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id']

First example ID:
56be4db0acb8001400a502ec


In [57]:

# Predict start and end logits for validation features
prediction_output = trainer.predict(tokenized_validation)

# Extract the two prediction arrays
start_logits, end_logits = prediction_output.predictions

print("Start logits shape:", start_logits.shape)
print("End logits shape:", end_logits.shape)


Start logits shape: (10784, 384)
End logits shape: (10784, 384)


In [58]:

import torch
import numpy as np
import collections

# Put the model in evaluation mode
model = trainer.model
model.eval()

# Generate predictions for validation features
predictions = trainer.predict(validation_features.remove_columns(
    ["example_id", "offset_mapping"]
))

# Extract start and end logits
start_logits = predictions.predictions[0]
end_logits = predictions.predictions[1]

print("Start logits shape:", start_logits.shape)
print("End logits shape:", end_logits.shape)


Start logits shape: (10784, 384)
End logits shape: (10784, 384)


In [72]:

import numpy as np

examples_by_id = {
    example["id"]: example
    for example in dataset["validation"]
}

example_predictions = {}

n_best = 20
max_answer_length = 30

for feature_index in range(len(validation_features)):

    example_id = validation_features[feature_index]["example_id"]

    context = examples_by_id[example_id]["context"]

    offsets = validation_features[feature_index]["offset_mapping"]

    start_logit = start_logits[feature_index]
    end_logit = end_logits[feature_index]

    # Get only valid context token positions
    context_token_indices = [
        i for i, offset in enumerate(offsets)
        if offset is not None and offset[1] > offset[0]
    ]

    if not context_token_indices:
        continue

    # Select top positions only from context tokens
    start_scores = start_logit[context_token_indices]
    end_scores = end_logit[context_token_indices]

    top_start_positions = np.argsort(start_scores)[-n_best:][::-1]
    top_end_positions = np.argsort(end_scores)[-n_best:][::-1]

    start_indexes = [
        context_token_indices[i] for i in top_start_positions
    ]

    end_indexes = [
        context_token_indices[i] for i in top_end_positions
    ]

    for start_index in start_indexes:

        for end_index in end_indexes:

            if end_index < start_index:
                continue

            if end_index - start_index + 1 > max_answer_length:
                continue

            start_char = offsets[start_index][0]
            end_char = offsets[end_index][1]

            predicted_text = context[start_char:end_char]

            score = (
                start_logit[start_index] +
                end_logit[end_index]
            )

            if (
                example_id not in example_predictions
                or score > example_predictions[example_id]["score"]
            ):

                example_predictions[example_id] = {
                    "score": float(score),
                    "text": predicted_text
                }

print("Questions with predictions:", len(example_predictions))
print("Total validation questions:", len(dataset["validation"]))


Questions with predictions: 10570
Total validation questions: 10570


In [82]:

# Prepare predictions in the required format
predictions = []

for example in dataset["validation"]:

    example_id = example["id"]

    predicted_answer = example_predictions.get(
        example_id, {"text": ""}
    )["text"]

    predictions.append({
        "id": example_id,
        "prediction_text": predicted_answer
    })

# Prepare actual answers
references = []

for example in dataset["validation"]:

    references.append({
        "id": example["id"],
        "answers": example["answers"]
    })

print("Predictions prepared:", len(predictions))
print("References prepared:", len(references))


Predictions prepared: 10570
References prepared: 10570


In [83]:

# Calculate SQuAD evaluation metrics
results = squad_metric.compute(
    predictions=predictions,
    references=references
)

print("Evaluation Results")
print("------------------")
print("Exact Match (EM):", results["exact_match"])
print("F1 Score:", results["f1"])


Evaluation Results
------------------
Exact Match (EM): 81.49479659413434
F1 Score: 88.65731607462968


In [84]:

for i in range(5):

    example = dataset["validation"][i]

    print("\nQuestion:", example["question"])

    print("Actual answer:", example["answers"]["text"])

    print(
        "Predicted answer:",
        predictions[i]["prediction_text"]
    )

    print("-" * 60)



Question: Which NFL team represented the AFC at Super Bowl 50?
Actual answer: ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
Predicted answer: Denver Broncos
------------------------------------------------------------

Question: Which NFL team represented the NFC at Super Bowl 50?
Actual answer: ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers']
Predicted answer: Carolina Panthers
------------------------------------------------------------

Question: Where did Super Bowl 50 take place?
Actual answer: ['Santa Clara, California', "Levi's Stadium", "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."]
Predicted answer: Santa Clara, California
------------------------------------------------------------

Question: Which NFL team won Super Bowl 50?
Actual answer: ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
Predicted answer: Denver Broncos
------------------------------------------------------------

Question: What color was used to em

In [85]:

import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

MODEL_PATH = "./fine_tuned_bert_squad"

# Load fine-tuned model and tokenizer
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Select GPU if available, otherwise CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("Fine-tuned BERT model loaded successfully!")
print("Device:", device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuned BERT model loaded successfully!
Device: cuda


In [86]:

def predict_answer(question, context):

    # Tokenize question and context
    inputs = tokenizer(
        question,
        context,
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    # Save character offsets before model prediction
    offsets = inputs.pop("offset_mapping")[0].tolist()

    # Identify tokens belonging to the context
    sequence_ids = inputs.sequence_ids(0)

    context_tokens = [
        i for i, sequence_id in enumerate(sequence_ids)
        if sequence_id == 1 and offsets[i][1] > offsets[i][0]
    ]

    # Move model inputs to GPU or CPU
    inputs = inputs.to(device)

    # Generate predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # Get start and end scores
    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    # Find the best valid answer span
    best_score = float("-inf")
    best_answer = ""

    for start_index in context_tokens:
        for end_index in context_tokens:

            # Answer cannot end before it starts
            if end_index < start_index:
                continue

            # Limit answer length
            if end_index - start_index + 1 > 30:
                continue

            score = (
                start_logits[start_index].item()
                + end_logits[end_index].item()
            )

            if score > best_score:
                best_score = score

                # Convert token positions into character positions
                start_char = offsets[start_index][0]
                end_char = offsets[end_index][1]

                # Extract answer from original context
                best_answer = context[start_char:end_char]

    return best_answer


In [87]:

question_1 = "Who was the first woman to win a Nobel Prize?"

context_1 = (
    "Marie Curie was a physicist and chemist who conducted "
    "pioneering research on radioactivity. She was the first "
    "woman to win a Nobel Prize and remains the only person "
    "to win Nobel Prizes in two different scientific fields."
)

answer_1 = predict_answer(question_1, context_1)

print("Question:", question_1)
print("Predicted Answer:", answer_1)
print("Expected Answer: Marie Curie")


Question: Who was the first woman to win a Nobel Prize?
Predicted Answer: Marie Curie
Expected Answer: Marie Curie


In [88]:

question_2 = "Where is the Eiffel Tower located?"

context_2 = (
    "The Eiffel Tower is a famous landmark located in Paris, "
    "France. It was designed by engineer Gustave Eiffel's "
    "company and completed in 1889. Millions of tourists "
    "visit this monument every year."
)

answer_2 = predict_answer(question_2, context_2)

print("Question:", question_2)
print("Predicted Answer:", answer_2)
print("Expected Answer: Paris, France")


Question: Where is the Eiffel Tower located?
Predicted Answer: Paris, France
Expected Answer: Paris, France


In [90]:

test_results = [
    {
        "question": question_1,
        "expected": "Marie Curie",
        "predicted": answer_1
    },
    {
        "question": question_2,
        "expected": "Paris, France",
        "predicted": answer_2
    }
]

for i, result in enumerate(test_results, start=1):

    print(f"\nCUSTOM TEST {i}")
    print("-" * 50)

    print("Question:", result["question"])
    print("Expected:", result["expected"])
    print("Predicted:", result["predicted"])

    print("-" * 50)



CUSTOM TEST 1
--------------------------------------------------
Question: Who was the first woman to win a Nobel Prize?
Expected: Marie Curie
Predicted: Marie Curie
--------------------------------------------------

CUSTOM TEST 2
--------------------------------------------------
Question: Where is the Eiffel Tower located?
Expected: Paris, France
Predicted: Paris, France
--------------------------------------------------


### Reflection

In this assignment, I successfully fine-tuned a pre-trained BERT model for extractive Question Answering using the SQuAD v1.1 dataset. I learned how to tokenize question-context pairs and map answer character positions to token positions. I used the Hugging Face Trainer API to train the model and evaluated its performance using Exact Match (EM) and F1 Score. The fine-tuned model achieved an Exact Match score of **81.49%** and an F1 Score of **88.66%** on the validation dataset. I also tested the model with two custom questions, and it correctly predicted both answers. Overall, this assignment helped me gain practical experience with transformer models, transfer learning, fine-tuning, and question-answering evaluation.